## Region Extraction

The MODIS NetCDF files contain **global** ocean data. To visualize a specific region (like the Red Sea), we need to extract only the data within our geographic bounds using xarray's `.sel()` method.


In [1]:
# How Region Extraction Works
# ============================
# 
# Step 1: Define geographic bounds (latitude and longitude)
# Step 2: Use xarray's .sel() method to slice the data
# Step 3: Only data within bounds is kept, rest is discarded

# Example: Inspect a dataset to see its structure
if len(files) > 0:
    import xarray as xr
    sample_file = os.path.join(data_dir, files[0])
    
    try:
        ds = xr.open_dataset(sample_file)
        print("📊 Dataset Structure:")
        print("=" * 60)
        print(f"Variables: {list(ds.data_vars)}")
        print(f"Coordinates: {list(ds.coords)}")
        print(f"Dimensions: {dict(ds.dims)}")
        print()
        
        # Show global extent
        if 'lat' in ds.coords and 'lon' in ds.coords:
            print("🌍 Global Data Extent:")
            print(f"   Latitude range: {float(ds.lat.min()):.2f}° to {float(ds.lat.max()):.2f}°")
            print(f"   Longitude range: {float(ds.lon.min()):.2f}° to {float(ds.lon.max()):.2f}°")
            print(f"   Total grid points: {len(ds.lat)} × {len(ds.lon)} = {len(ds.lat) * len(ds.lon):,}")
            print()
            
            # Show region extraction
            print("🔍 Region Extraction Process:")
            print(f"   Target region: Red Sea")
            print(f"   Latitude: {lat_min}°N to {lat_max}°N")
            print(f"   Longitude: {lon_min}°E to {lon_max}°E")
            print()
            
            # Extract region
            if 'chlor_a' in ds.data_vars:
                chl_global = ds['chlor_a']
                chl_region = chl_global.sel(lat=slice(lat_min, lat_max), 
                                           lon=slice(lon_min, lon_max))
                
                print("   Before extraction:")
                print(f"      Data shape: {chl_global.shape}")
                print(f"      Data size: {chl_global.size:,} values")
                print()
                print("   After extraction:")
                print(f"      Data shape: {chl_region.shape}")
                print(f"      Data size: {chl_region.size:,} values")
                print(f"      Reduction: {chl_region.size/chl_global.size*100:.1f}% of original")
            
        ds.close()
        
    except Exception as e:
        print(f"⚠️  Could not inspect file: {str(e)}")
        print("   Make sure you have valid NetCDF files in the data directory")
else:
    print("⚠️  No data files found. Download data first to see region extraction in action.")


NameError: name 'files' is not defined

### How `.sel()` Works

The `.sel()` method uses **slicing** to extract data:

```python
# Extract region using slice
chl_region = ds['chlor_a'].sel(
    lat=slice(lat_min, lat_max),  # Latitude slice: 12°N to 30°N
    lon=slice(lon_min, lon_max)    # Longitude slice: 32°E to 44°E
)
```

**What happens:**
1. `lat=slice(12, 30)` keeps only latitudes between 12°N and 30°N
2. `lon=slice(32, 44)` keeps only longitudes between 32°E and 44°E
3. All data outside these bounds is discarded
4. Result: Only Red Sea data remains

**Benefits:**
- ✅ Faster processing (less data to handle)
- ✅ Smaller memory usage
- ✅ Focused visualization
- ✅ Better performance for animations


In [ ]:
# Change Region: Customize for Your Area
# =======================================
# 
# To visualize a different region, modify these bounds:

# Current: Red Sea
# lat_min, lat_max = 12, 30
# lon_min, lon_max = 32, 44

# Examples of other regions:

# Mediterranean Sea
# lat_min, lat_max = 30, 46
# lon_min, lon_max = -6, 36

# Gulf of Mexico
# lat_min, lat_max = 18, 31
# lon_min, lon_max = -98, -80

# Arabian Sea
# lat_min, lat_max = 5, 25
# lon_min, lon_max = 50, 75

# North Atlantic
# lat_min, lat_max = 20, 50
# lon_min, lon_max = -80, -10

# To use a different region:
# 1. Uncomment one of the examples above (or define your own)
# 2. Comment out the Red Sea bounds
# 3. Re-run the visualization cells

print("Region bounds configured:")
print(f"  Latitude: {lat_min}°N to {lat_max}°N")
print(f"  Longitude: {lon_min}°E to {lon_max}°E")
print(f"\nTo change region, edit the bounds in the configuration cell above.")


### Visual Explanation

```
Global MODIS Data (Full World Coverage)
┌─────────────────────────────────────────┐
│                                         │
│     ┌──────────────────────┐           │
│     │                      │           │
│     │   Red Sea Region     │  ← Extract this
│     │   (12-30°N, 32-44°E) │           │
│     │                      │           │
│     └──────────────────────┘           │
│                                         │
└─────────────────────────────────────────┘
         ↓ .sel() extraction
         
Red Sea Data Only
┌──────────────────────┐
│                      │
│   Red Sea Region     │  ← Only this remains
│   (12-30°N, 32-44°E) │
│                      │
└──────────────────────┘
```


## Download Data

Before visualizing, you need to download MODIS data files. Choose one of the options below.


### Option A: Quick Download (Recommended)

Download a few sample files to test the visualization. This will download 3-4 files from 2023.


In [ ]:
# Quick download function - downloads sample files
import requests
from datetime import datetime, timedelta

def download_sample_files(num_files=4):
    """
    Download a few sample MODIS files for testing.
    Downloads files from 2023 (known to have data available).
    Note: Some files may require Earthdata authentication.
    """
    base_url = "https://oceandata.sci.gsfc.nasa.gov/cgi/getfile/"
    os.makedirs(data_dir, exist_ok=True)
    
    # Use dates from 2023 that are known to have data
    base_date = datetime(2023, 6, 1)  # June 2023
    downloaded = 0
    failed = 0
    
    print(f"Downloading {num_files} sample files from 2023...")
    print("=" * 60)
    print("⚠️  Note: Some files may require Earthdata login.")
    print("   If downloads fail, try manual download from:")
    print("   https://oceandata.sci.gsfc.nasa.gov/")
    print("=" * 60)
    
    for i in range(num_files):
        # Calculate 8-day period
        period_start = base_date + timedelta(days=i*8)
        period_end = period_start + timedelta(days=7)
        
        date_str = period_start.strftime("%Y%m%d")
        end_date_str = period_end.strftime("%Y%m%d")
        
        filename = f"AQUA_MODIS.{date_str}_{end_date_str}.L3m.8D.CHL.chlor_a.4km.nc"
        url = base_url + filename
        output_path = os.path.join(data_dir, filename)
        
        # Check if file exists and is valid (not HTML)
        if os.path.exists(output_path):
            file_size = os.path.getsize(output_path)
            if file_size > 100000:  # Valid NetCDF should be > 100KB
                print(f"✓ {filename} already exists and is valid")
                downloaded += 1
                continue
            else:
                # Remove invalid HTML file
                os.remove(output_path)
        
        try:
            print(f"\nDownloading {filename}...")
            response = requests.get(url, stream=True, timeout=30)
            
            if response.status_code == 200:
                # Check if we got HTML instead of NetCDF
                first_chunk = b''
                total_size = int(response.headers.get('content-length', 0))
                
                with open(output_path, 'wb') as f:
                    if total_size > 0:
                        with tqdm(total=total_size, unit='B', unit_scale=True, 
                                 desc=filename[:50]) as pbar:
                            for chunk in response.iter_content(chunk_size=8192):
                                if chunk:
                                    if not first_chunk:
                                        first_chunk = chunk[:200]
                                    f.write(chunk)
                                    pbar.update(len(chunk))
                    else:
                        for chunk in response.iter_content(chunk_size=8192):
                            if chunk:
                                if not first_chunk:
                                    first_chunk = chunk[:200]
                                f.write(chunk)
                
                # Validate file
                if first_chunk.startswith(b'<') or b'html' in first_chunk.lower() or b'earthdata' in first_chunk.lower():
                    os.remove(output_path)
                    print(f"  ✗ Got HTML error page (may require authentication)")
                    failed += 1
                elif os.path.getsize(output_path) < 100000:
                    os.remove(output_path)
                    print(f"  ✗ File too small, likely invalid")
                    failed += 1
                else:
                    print(f"  ✓ Downloaded: {filename} ({os.path.getsize(output_path)/1024/1024:.1f} MB)")
                    downloaded += 1
            else:
                print(f"  ✗ Failed: {filename} (HTTP {response.status_code})")
                failed += 1
        except Exception as e:
            print(f"  ✗ Error downloading {filename}: {str(e)}")
            if os.path.exists(output_path):
                os.remove(output_path)
            failed += 1
    
    print("\n" + "=" * 60)
    print(f"Download complete: {downloaded} successful, {failed} failed")
    
    if downloaded > 0:
        print(f"\n✅ Sample files downloaded to {data_dir}/")
        print("   You can now run the visualization cells below!")
    else:
        print("\n⚠️  All downloads failed. This usually means:")
        print("   1. Files require Earthdata authentication")
        print("   2. Files don't exist at those dates")
        print("   3. Network/server issues")
        print("\n💡 Solutions:")
        print("   • Create free Earthdata account: https://urs.earthdata.nasa.gov/")
        print("   • Download manually from: https://oceandata.sci.gsfc.nasa.gov/")
        print("   • Try different dates (2022-2023 typically work)")

# Uncomment the line below to download sample files:
# download_sample_files(num_files=4)


### Option B: Download Specific Date Range

Use the `down_files.py` script to download data for a specific time period.


In [ ]:
# Option B: Download using the script
# Run this in terminal: python down_files.py
# Or uncomment below to run it from the notebook:

# import subprocess
# subprocess.run(['python', 'down_files.py'])


### Option C: Manual Download (Recommended if automated fails)

**Many MODIS files now require Earthdata authentication.**

1. **Create free account**: https://urs.earthdata.nasa.gov/ (takes 2 minutes)
2. **Visit**: https://oceandata.sci.gsfc.nasa.gov/
3. **Login** with your Earthdata credentials
4. **Navigate to**: MODIS-Aqua → Level 3 → Chlorophyll-a
5. **Select**: 
   - Temporal: 8-day composite
   - Spatial: 4km
   - Dates: Try 2022-2023 (known to have data)
6. **Download** files and place in `data/` directory

**File naming format:**
`AQUA_MODIS.YYYYMMDD_YYYYMMDD.L3m.8D.CHL.chlor_a.4km.nc`

**Alternative: Use ERDDAP (no login required)**
- Visit: https://coastwatch.pfeg.noaa.gov/erddap/
- Search for "MODIS chlorophyll"
- Download as NetCDF


## Troubleshooting: If imageio is not working

If you get import errors, the Jupyter kernel might be using a different Python environment.

**Quick fix:**
1. Run this in a cell: `!pip install imageio imageio-ffmpeg`
2. Restart the kernel (Kernel → Restart)
3. Re-run the import cell

**Or check your environment:**
- Run: `python check_environment.py` from terminal
- This will show which Python Jupyter is using and install missing packages


In [ ]:
# Configuration parameters
data_dir = 'data/' 
output_dir = 'frames/'
animation_output = 'chlorophyll_animation.mp4'

# Create output directory if it doesn't exist
os.makedirs(output_dir, exist_ok=True)

# Red Sea geographic bounds
lat_min, lat_max = 12, 30
lon_min, lon_max = 32, 44

# Visualization parameters
cmap = 'turbo'  # Color map for chlorophyll
vmin, vmax = 0.05, 5.0  # Chlorophyll-a concentration range (mg/m³)
dpi = 150  # Resolution for saved frames

print(f"Data directory: {data_dir}")
print(f"Output directory: {output_dir}")
print(f"Region: Red Sea ({lat_min}°N to {lat_max}°N, {lon_min}°E to {lon_max}°E)")


Data directory: data/
Output directory: frames/
Region: Red Sea (12°N to 30°N, 32°E to 44°E)


In [ ]:
# List all NetCDF files in data directory
files = sorted([f for f in os.listdir(data_dir) if f.endswith('.nc')])

if len(files) == 0:
    print("⚠️  No .nc files found in data directory!")
    print("Please download data files first using down_files.py")
else:
    print(f"Found {len(files)} NetCDF files")
    print(f"First file: {files[0]}")
    print(f"Last file: {files[-1]}")


⚠️  No .nc files found in data directory!
Please download data files first using down_files.py


In [ ]:
# Function to extract time from dataset
def get_time_string(ds):
    """Extract time string from xarray dataset."""
    try:
        if 'time' in ds.coords:
            time_val = ds.time.values
            if hasattr(time_val, '__len__') and len(time_val) > 0:
                time_str = str(time_val[0])[:10]
            else:
                time_str = str(time_val)[:10]
        else:
            # Try to get from filename
            time_str = "Unknown"
        return time_str
    except:
        return "Unknown"

# Function to create a single frame
def create_frame(filepath, output_path, frame_num):
    """
    Create a visualization frame from a NetCDF file.
    
    Args:
        filepath: Path to NetCDF file
        output_path: Path to save the frame
        frame_num: Frame number for progress tracking
    
    Returns:
        True if successful, False otherwise
    """
    try:
        # Open dataset
        ds = xr.open_dataset(filepath)
        
        # Check if chlor_a variable exists
        if 'chlor_a' not in ds.data_vars:
            print(f"⚠️  Warning: 'chlor_a' not found in {os.path.basename(filepath)}")
            # Try alternative variable names
            alt_names = ['chlorophyll', 'CHL', 'chlor']
            found = False
            for alt in alt_names:
                if alt in ds.data_vars:
                    chl = ds[alt].sel(lat=slice(lat_min, lat_max), lon=slice(lon_min, lon_max))
                    found = True
                    break
            if not found:
                return False
        else:
            # Select region of interest
            chl = ds['chlor_a'].sel(lat=slice(lat_min, lat_max), lon=slice(lon_min, lon_max))
        
        # Get time information
        time_str = get_time_string(ds)
        
        # Create figure with cartopy projection
        fig = plt.figure(figsize=(10, 8))
        ax = plt.axes(projection=ccrs.PlateCarree())
        
        # Plot chlorophyll data
        im = chl.plot(ax=ax, transform=ccrs.PlateCarree(),
                     cmap=cmap, vmin=vmin, vmax=vmax,
                     add_colorbar=False)
        
        # Add colorbar
        cbar = plt.colorbar(im, ax=ax, orientation='horizontal', 
                           pad=0.05, aspect=40, shrink=0.8)
        cbar.set_label('Chlorophyll-a (mg/m³)', fontsize=12)
        
        # Set map extent and add features
        ax.set_extent([lon_min, lon_max, lat_min, lat_max], crs=ccrs.PlateCarree())
        ax.add_feature(cfeature.COASTLINE, linewidth=0.8, color='black')
        ax.add_feature(cfeature.BORDERS, linewidth=0.5, linestyle='--', alpha=0.5)
        ax.add_feature(cfeature.LAND, facecolor='lightgray', alpha=0.5)
        
        # Add gridlines
        gl = ax.gridlines(draw_labels=True, linewidth=0.5, color='gray', 
                         alpha=0.5, linestyle='--')
        gl.top_labels = False
        gl.right_labels = False
        
        # Set title
        ax.set_title(f"MODIS Chlorophyll-a Concentration — {time_str}", 
                    fontsize=14, fontweight='bold', pad=15)
        
        # Save figure
        plt.tight_layout()
        plt.savefig(output_path, dpi=dpi, bbox_inches='tight', facecolor='white')
        plt.close()
        
        # Close dataset
        ds.close()
        
        return True
        
    except Exception as e:
        print(f"❌ Error processing {os.path.basename(filepath)}: {str(e)}")
        return False

print("Frame creation function defined!")


Frame creation function defined!


In [ ]:
# Generate frames from all NetCDF files
if len(files) > 0:
    successful_frames = []
    failed_files = []
    
    print("Generating frames...")
    for i, f in enumerate(tqdm(files, desc="Processing files")):
        filepath = os.path.join(data_dir, f)
        frame_path = os.path.join(output_dir, f'frame_{i:03d}.png')
        
        if create_frame(filepath, frame_path, i):
            successful_frames.append(frame_path)
        else:
            failed_files.append(f)
    
    print(f"\n✅ Successfully created {len(successful_frames)} frames")
    if failed_files:
        print(f"⚠️  Failed to process {len(failed_files)} files:")
        for f in failed_files:
            print(f"   - {f}")
else:
    print("No files to process. Please download data first.")


No files to process. Please download data first.


In [ ]:
# Create animation from frames
def create_animation(frame_paths, output_path, fps=2):
    """
    Create an MP4 animation from a list of frame images.
    
    Args:
        frame_paths: List of paths to frame images
        output_path: Path to save the animation
        fps: Frames per second for the animation
    """
    if len(frame_paths) == 0:
        print("No frames to animate!")
        return False
    
    # Check if imageio is available
    if not IMAGEIO_AVAILABLE:
        print("❌ imageio is not installed. Cannot create MP4 animation.")
        print("\nTo fix this:")
        print("   1. Make sure you're installing in the correct Python environment")
        print("   2. Install imageio: pip install imageio imageio-ffmpeg")
        print("   3. If using Jupyter, you may need to restart the kernel after installation")
        print("\nAlternative: Use matplotlib animation (see cell below)")
        return False
    
    print(f"Creating animation from {len(frame_paths)} frames...")
    
    try:
        # Read all frames
        images = []
        for frame_path in tqdm(frame_paths, desc="Loading frames"):
            if os.path.exists(frame_path):
                images.append(imageio.imread(frame_path))
        
        if len(images) == 0:
            print("No valid frames found!")
            return False
        
        # Save as MP4
        imageio.mimsave(output_path, images, fps=fps, codec='libx264', 
                       quality=8, pixelformat='yuv420p')
        
        print(f"✅ Animation saved to: {output_path}")
        return True
        
    except Exception as e:
        print(f"❌ Error creating animation: {str(e)}")
        print("\nTroubleshooting:")
        print("   1. Install imageio-ffmpeg: pip install imageio-ffmpeg")
        print("   2. If using conda: conda install -c conda-forge imageio-ffmpeg")
        print("   3. Restart Jupyter kernel after installation")
        print("   4. Check if ffmpeg is installed on your system")
        return False

# Alternative: Create animation using matplotlib (no imageio required)
def create_animation_matplotlib(frame_paths, output_path, fps=2):
    """
    Create an animation using matplotlib's FuncAnimation.
    This doesn't require imageio, but creates a GIF instead of MP4.
    """
    from matplotlib.animation import FuncAnimation, PillowWriter
    import matplotlib.image as mpimg
    
    if len(frame_paths) == 0:
        print("No frames to animate!")
        return False
    
    print(f"Creating GIF animation from {len(frame_paths)} frames using matplotlib...")
    
    try:
        # Read first frame to get dimensions
        first_img = mpimg.imread(frame_paths[0])
        fig, ax = plt.subplots(figsize=(10, 8))
        ax.axis('off')
        
        im = ax.imshow(first_img)
        
        def animate(frame_num):
            img = mpimg.imread(frame_paths[frame_num])
            im.set_array(img)
            return [im]
        
        anim = FuncAnimation(fig, animate, frames=len(frame_paths), 
                            interval=1000/fps, blit=True, repeat=True)
        
        # Save as GIF
        gif_path = output_path.replace('.mp4', '.gif')
        writer = PillowWriter(fps=fps)
        anim.save(gif_path, writer=writer)
        plt.close()
        
        print(f"✅ GIF animation saved to: {gif_path}")
        return True
        
    except Exception as e:
        print(f"❌ Error creating matplotlib animation: {str(e)}")
        return False

# Create animation if frames were generated
if len(files) > 0 and 'successful_frames' in locals() and len(successful_frames) > 0:
    # Try imageio first, fallback to matplotlib
    if not create_animation(successful_frames, animation_output, fps=2):
        print("\nTrying alternative method with matplotlib...")
        create_animation_matplotlib(successful_frames, animation_output, fps=2)
else:
    print("No frames available for animation.")


No frames available for animation.


In [ ]:
# Optional: Display a sample frame
import matplotlib.image as mpimg

if len(files) > 0:
    sample_frame = os.path.join(output_dir, 'frame_000.png')
    if os.path.exists(sample_frame):
        img = mpimg.imread(sample_frame)
        plt.figure(figsize=(12, 8))
        plt.imshow(img)
        plt.axis('off')
        plt.title('Sample Frame', fontsize=14, fontweight='bold')
        plt.tight_layout()
        plt.show()
    else:
        print("No sample frame found. Run the frame generation cell first.")


## Kepler.gl Integration

The following cells show how to export data for use with Kepler.gl, a powerful web-based geospatial visualization tool.


## Export to Kepler.gl - Process All Files

This improved export function processes **ALL** your data files and creates an optimized CSV for Kepler.gl visualization.


In [ ]:
# Improved Kepler.gl Export - Processes ALL Files
# ===============================================

def export_all_to_kepler(data_dir, output_file='kepler_chlorophyll_all.csv', sample_rate=3):
    """
    Export ALL chlorophyll-a data files to CSV format for Kepler.gl.
    
    This function processes all NetCDF files and creates a CSV optimized for Kepler.gl
    with time-series animation support.
    
    Args:
        data_dir: Directory containing NetCDF files
        output_file: Output CSV filename
        sample_rate: Sample every Nth point (3 = every 3rd point) to reduce file size
                    Lower = more detail but larger file. 3-5 is recommended.
    
    Returns:
        DataFrame with exported data
    """
    import pandas as pd
    import json
    from datetime import datetime
    
    # Ensure data_dir exists and get files
    if not os.path.exists(data_dir):
        print(f"❌ Data directory not found: {data_dir}")
        return None
    
    files = sorted([f for f in os.listdir(data_dir) if f.endswith('.nc')])
    
    if len(files) == 0:
        print(f"❌ No .nc files found in data directory: {data_dir}")
        print(f"   Current directory: {os.getcwd()}")
        print(f"   Files in data_dir: {os.listdir(data_dir) if os.path.exists(data_dir) else 'directory does not exist'}")
        return None
    
    all_data = []
    
    print(f"📊 Processing {len(files)} files for Kepler.gl export...")
    print(f"   Sampling rate: every {sample_rate} points (to manage file size)")
    print("=" * 60)
    
    for i, f in enumerate(tqdm(files, desc="Exporting files"), 1):
        try:
            # Open dataset
            ds = xr.open_dataset(os.path.join(data_dir, f))
            
            # Get chlorophyll data
            if 'chlor_a' not in ds.data_vars:
                # Try alternative names
                alt_names = ['chlorophyll', 'CHL', 'chlor']
                chl = None
                for alt in alt_names:
                    if alt in ds.data_vars:
                        chl = ds[alt].sel(lat=slice(lat_min, lat_max), 
                                         lon=slice(lon_min, lon_max))
                        break
                if chl is None:
                    ds.close()
                    continue
            else:
                chl = ds['chlor_a'].sel(lat=slice(lat_min, lat_max), 
                                        lon=slice(lon_min, lon_max))
            
            # Get time information
            time_str = get_time_string(ds)
            
            # Sample data to reduce file size (every Nth point)
            chl_sampled = chl[::sample_rate, ::sample_rate]
            
            # Convert to DataFrame
            df = chl_sampled.to_dataframe().reset_index()
            df = df[['lat', 'lon', 'chlor_a']].dropna()
            df.columns = ['lat', 'lon', 'value']
            df['time'] = time_str
            df['date'] = pd.to_datetime(time_str)
            df['timestamp'] = pd.to_datetime(time_str).timestamp()  # For Kepler.gl time filter
            
            all_data.append(df)
            ds.close()
            
        except Exception as e:
            print(f"⚠️  Error processing {f}: {str(e)}")
            continue
    
    if len(all_data) == 0:
        print("❌ No data extracted!")
        return None
    
    # Combine all dataframes
    combined_df = pd.concat(all_data, ignore_index=True)
    
    # Sort by date for better time-series visualization
    combined_df = combined_df.sort_values('date')
    
    # Save to CSV
    combined_df.to_csv(output_file, index=False)
    
    print("\n" + "=" * 60)
    print(f"✅ Data exported to: {output_file}")
    print(f"   Total records: {len(combined_df):,}")
    print(f"   Unique dates: {combined_df['date'].nunique()}")
    print(f"   Date range: {combined_df['date'].min().strftime('%Y-%m-%d')} to {combined_df['date'].max().strftime('%Y-%m-%d')}")
    print(f"   File size: {os.path.getsize(output_file) / 1024 / 1024:.1f} MB")
    print("\n" + "=" * 60)
    print("📋 How to use in Kepler.gl:")
    print("   1. Go to: https://kepler.gl/")
    print("   2. Click 'Add Data' → 'Upload File'")
    print("   3. Upload:", output_file)
    print("   4. Configure layer:")
    print("      • Layer type: 'Heatmap' or 'Point'")
    print("      • Color by: 'value' (chlorophyll-a)")
    print("      • Time: 'date' (for animation)")
    print("   5. Use time slider to animate through dates!")
    print("=" * 60)
    
    return combined_df

# Export all your data files to Kepler.gl format
# Adjust sample_rate: lower = more detail (larger file), higher = less detail (smaller file)

# Make sure data_dir is defined (from configuration cell)
if 'data_dir' not in locals():
    data_dir = 'data/'

# Check if files exist first
files_check = sorted([f for f in os.listdir(data_dir) if f.endswith('.nc')]) if os.path.exists(data_dir) else []
print(f"Found {len(files_check)} .nc files in {data_dir}")

if len(files_check) > 0:
    kepler_data = export_all_to_kepler(data_dir, 'kepler_chlorophyll_all.csv', sample_rate=3)
else:
    print("⚠️  No data files found. Make sure you've downloaded files to the data/ directory.")


### Quick Preview of Exported Data

Check the exported data structure:


In [ ]:
# Preview the exported data
if 'kepler_data' in locals() and kepler_data is not None:
    print("📊 Exported Data Preview:")
    print("=" * 60)
    print(kepler_data.head(10))
    print("\n...")
    print(f"\nTotal rows: {len(kepler_data):,}")
    print(f"\nColumn info:")
    print(kepler_data.info())
    print("\n📈 Value statistics:")
    print(kepler_data['value'].describe())
else:
    print("⚠️  Run the export cell above first!")


In [ ]:
# Export data to GeoJSON/CSV for Kepler.gl
import pandas as pd
import json
from datetime import datetime

def export_to_kepler(data_dir, output_file='kepler_data.csv'):
    """
    Export chlorophyll-a data to CSV format compatible with Kepler.gl.
    
    Creates a CSV file with columns: lat, lon, value, time
    This can be imported into Kepler.gl for interactive visualization.
    """
    files = sorted([f for f in os.listdir(data_dir) if f.endswith('.nc')])
    
    if len(files) == 0:
        print("No data files found!")
        return None
    
    all_data = []
    
    print(f"Processing {len(files)} files for Kepler.gl export...")
    
    for f in tqdm(files, desc="Exporting data"):
        try:
            ds = xr.open_dataset(os.path.join(data_dir, f))
            
            # Get chlorophyll data
            if 'chlor_a' in ds.data_vars:
                chl = ds['chlor_a'].sel(lat=slice(lat_min, lat_max), 
                                        lon=slice(lon_min, lon_max))
            else:
                # Try alternative names
                alt_names = ['chlorophyll', 'CHL', 'chlor']
                chl = None
                for alt in alt_names:
                    if alt in ds.data_vars:
                        chl = ds[alt].sel(lat=slice(lat_min, lat_max), 
                                         lon=slice(lon_min, lon_max))
                        break
                
                if chl is None:
                    ds.close()
                    continue
            
            # Get time
            time_str = get_time_string(ds)
            
            # Convert to DataFrame
            df = chl.to_dataframe().reset_index()
            df = df[['lat', 'lon', 'chlor_a']].dropna()
            df.columns = ['lat', 'lon', 'value']
            df['time'] = time_str
            df['date'] = pd.to_datetime(time_str)
            
            all_data.append(df)
            ds.close()
            
        except Exception as e:
            print(f"Error processing {f}: {str(e)}")
            continue
    
    if len(all_data) == 0:
        print("No data extracted!")
        return None
    
    # Combine all dataframes
    combined_df = pd.concat(all_data, ignore_index=True)
    
    # Save to CSV
    combined_df.to_csv(output_file, index=False)
    print(f"\n✅ Data exported to {output_file}")
    print(f"   Total records: {len(combined_df)}")
    print(f"   Date range: {combined_df['date'].min()} to {combined_df['date'].max()}")
    print(f"\nTo use in Kepler.gl:")
    print(f"   1. Go to https://kepler.gl/")
    print(f"   2. Click 'Add Data' and upload {output_file}")
    print(f"   3. Configure layers with:")
    print(f"      - Layer type: Heatmap or Point")
    print(f"      - Color by: value")
    print(f"      - Time: date (for animation)")
    
    return combined_df

# Uncomment to export data for Kepler.gl
# kepler_data = export_to_kepler(data_dir, 'kepler_chlorophyll_data.csv')


In [ ]:
# Alternative: Create a simplified GeoJSON for Kepler.gl
def export_geojson_sample(data_dir, output_file='kepler_data.geojson', sample_rate=10):
    """
    Export a sample of data points as GeoJSON for Kepler.gl.
    
    Args:
        data_dir: Directory containing NetCDF files
        output_file: Output GeoJSON filename
        sample_rate: Sample every Nth point (to reduce file size)
    """
    files = sorted([f for f in os.listdir(data_dir) if f.endswith('.nc')])
    
    if len(files) == 0:
        print("No data files found!")
        return None
    
    features = []
    
    print(f"Creating GeoJSON from {len(files)} files (sampling every {sample_rate} points)...")
    
    for f in tqdm(files[:5], desc="Processing"):  # Limit to first 5 files for demo
        try:
            ds = xr.open_dataset(os.path.join(data_dir, f))
            
            if 'chlor_a' not in ds.data_vars:
                ds.close()
                continue
            
            chl = ds['chlor_a'].sel(lat=slice(lat_min, lat_max), 
                                   lon=slice(lon_min, lon_max))
            time_str = get_time_string(ds)
            
            # Sample data points
            chl_sampled = chl[::sample_rate, ::sample_rate]
            
            # Convert to numpy arrays
            lats = chl_sampled.lat.values
            lons = chl_sampled.lon.values
            values = chl_sampled.values
            
            # Create GeoJSON features
            for i in range(len(lats)):
                for j in range(len(lons)):
                    if not np.isnan(values[i, j]):
                        feature = {
                            "type": "Feature",
                            "geometry": {
                                "type": "Point",
                                "coordinates": [float(lons[j]), float(lats[i])]
                            },
                            "properties": {
                                "value": float(values[i, j]),
                                "time": time_str,
                                "date": time_str
                            }
                        }
                        features.append(feature)
            
            ds.close()
            
        except Exception as e:
            print(f"Error processing {f}: {str(e)}")
            continue
    
    geojson = {
        "type": "FeatureCollection",
        "features": features
    }
    
    with open(output_file, 'w') as f:
        json.dump(geojson, f)
    
    print(f"\n✅ GeoJSON exported to {output_file}")
    print(f"   Total features: {len(features)}")
    
    return geojson



In [ ]:
geojson_data = export_geojson_sample(data_dir, 'kepler_chlorophyll.geojson', sample_rate=5)

No data files found!
